In [ ]:
!pip -q install rtdl_num_embeddings scikit-learn pandas tabm

import os, sys, time, json, copy, math, warnings
import numpy as np, pandas as pd, torch, torch.nn.functional as F
warnings.filterwarnings('ignore')

sys.path.insert(0, '/content')          # tabm.py uploaded here
from tabm import TabM

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '| GPU:', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'NONE (will be slow)')

# Optional: save results to Google Drive so they survive a disconnect
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/tabm_grid'
else:
    OUT_DIR = '/content/tabm_grid'
os.makedirs(OUT_DIR, exist_ok=True)
RESULTS_CSV = f'{OUT_DIR}/grid_results.csv'
META_CSV = f'{OUT_DIR}/meta_features.csv'

Device: cuda | GPU: Tesla T4
Mounted at /content/drive


In [ ]:
!pip -q install optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---- Search space (sampled per dataset by TPE) ----
#   k: 4..64 (powers of 2) | depth: 1..6 | width: 64..1024 (powers of 2)
#   lr: 3e-4..1e-2 (log)   | dropout: 0..0.4 | weight_decay: 1e-6..1e-2 (log)
N_TRIALS  = 30     # per dataset; pruned trials count toward this
N_STARTUP = 8      # random trials before TPE kicks in
FINAL_SEEDS = [0]  # one seed
DROPOUT      = 0.1   # defaults (also used by the default-config trial)
WEIGHT_DECAY = 3e-4
DEFAULT_PARAMS = dict(log2_k=5, depth=3, log2_width=9, lr=2e-3, dropout=0.1, wd=3e-4)  # k=32, depth=3, width=512
MAX_EPOCHS   = 50
PATIENCE     = 8
MAX_TRAIN    = 10_000
MAX_EVAL     = 5_000

RESULTS_CSV = f'{OUT_DIR}/bo_results.csv'        # new file, your grid_results.csv stays untouched
STORAGE = f'sqlite:///{OUT_DIR}/optuna.db'       # lets the search resume after a disconnect

# ---- Datasets: (name, source, task) ----
DATASETS = [
    ('california',     ('sklearn', None),              'regression'),
    ('house_16H',      ('openml', 'house_16H', 1),     'regression'),
    ('diamonds',       ('openml', 'diamonds', 1),      'regression'),
    ('adult',          ('openml', 'adult', 2),         'binclass'),
    ('MagicTelescope', ('openml', 'MagicTelescope', 1),'binclass'),
    ('phoneme',        ('openml', 'phoneme', 1),       'binclass'),
    ('satimage',       ('openml', 'satimage', 1),      'multiclass'),
]

print(f'up to {N_TRIALS} trials x {len(DATASETS)} datasets = {N_TRIALS*len(DATASETS)} runs (pruned ones stop early)')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 26.3 MB/s eta 0:00:00
up to 30 trials x 7 datasets = 210 runs (pruned ones stop early)


In [ ]:
from sklearn.datasets import fetch_openml, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer, LabelEncoder

def load_raw(source):
    if source[0] == 'sklearn':
        d = fetch_california_housing(as_frame=True)
        return d.data, d.target
    _, name, version = source
    d = fetch_openml(name=name, version=version, as_frame=True, parser='auto')
    return d.data, d.target

def prepare(name, source, task, seed=0):
    X, y = load_raw(source)
    X = X.copy()

    # Labels
    if task == 'regression':
        y = pd.to_numeric(y, errors='coerce').astype(float)
        keep = y.notna(); X, y = X[keep], y[keep]
    else:
        y = pd.Series(LabelEncoder().fit_transform(y.astype(str)), index=X.index)

    # Column types
    cat_cols = [c for c in X.columns if X[c].dtype.name in ('category', 'object', 'bool')]
    num_cols = [c for c in X.columns if c not in cat_cols]

    # Splits: 64% train / 16% val / 20% test
    strat = y if task != 'regression' else None
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=strat)
    strat = y_tr if task != 'regression' else None
    X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=0.2, random_state=seed, stratify=strat)
    X_tr, y_tr = X_tr.iloc[:MAX_TRAIN], y_tr.iloc[:MAX_TRAIN]
    X_va, y_va = X_va.iloc[:MAX_EVAL], y_va.iloc[:MAX_EVAL]
    X_te, y_te = X_te.iloc[:MAX_EVAL], y_te.iloc[:MAX_EVAL]

    # Numeric features: median-impute, then quantile-normalize (as in the TabM paper)
    qt = None
    if num_cols:
        med = X_tr[num_cols].apply(pd.to_numeric, errors='coerce').median()
        def num(df): return df[num_cols].apply(pd.to_numeric, errors='coerce').fillna(med).values.astype(np.float32)
        a_tr = num(X_tr)
        noise = np.random.default_rng(seed).normal(0, 1e-5, a_tr.shape).astype(np.float32)
        qt = QuantileTransformer(output_distribution='normal', subsample=10**9,
                                 n_quantiles=max(min(len(a_tr) // 30, 1000), 10), random_state=seed)
        qt.fit(a_tr + noise * a_tr.std(0, keepdims=True))

    # Categorical features: integer codes, 0 = missing/unseen
    cat_maps = {c: {v: i + 1 for i, v in enumerate(X_tr[c].astype(str).unique())} for c in cat_cols}
    card = [len(cat_maps[c]) + 1 for c in cat_cols]
    def cat(df):
        return np.stack([df[c].astype(str).map(cat_maps[c]).fillna(0).astype(np.int64).values
                         for c in cat_cols], 1)

    # Regression target standardization (RMSE is reported in original units)
    y_mean, y_std = (float(y_tr.mean()), float(y_tr.std())) if task == 'regression' else (0.0, 1.0)

    D = {'name': name, 'task': task, 'n_num': len(num_cols), 'card': card,
         'd_out': 1 if task == 'regression' else int(y.nunique()),
         'y_mean': y_mean, 'y_std': y_std}
    for part, Xp, yp in [('train', X_tr, y_tr), ('val', X_va, y_va), ('test', X_te, y_te)]:
        xn = torch.tensor(qt.transform(num(Xp)), dtype=torch.float32, device=device) if num_cols else None
        xc = torch.tensor(cat(Xp), device=device) if cat_cols else None
        if task == 'regression':
            yt = torch.tensor(((yp.values - y_mean) / y_std).astype(np.float32), device=device)
        else:
            yt = torch.tensor(yp.values.astype(np.int64), device=device)
        D[part] = {'x_num': xn, 'x_cat': xc, 'y': yt}

    # Meta-features (used later by the selectors)
    n_tr = len(y_tr)
    meta = {'dataset': name, 'task': task, 'n_train': n_tr, 'n_num': len(num_cols),
            'n_cat': len(cat_cols), 'sum_card': int(sum(card)), 'n_classes': D['d_out'] if task != 'regression' else 0,
            'majority_frac': float(y_tr.value_counts(normalize=True).max()) if task != 'regression' else np.nan,
            'target_skew': float(y_tr.skew()) if task == 'regression' else np.nan}
    return D, meta

In [ ]:
def batch_size_for(n):  # dataset-size-based batch size, roughly like the paper
    return 128 if n < 10_000 else 256 if n < 30_000 else 512 if n < 100_000 else 1024

def build_model(D, k, depth, width, dropout=DROPOUT):
    return TabM.make(n_num_features=D['n_num'], cat_cardinalities=D['card'], d_out=D['d_out'],
                     k=k, n_blocks=depth, d_block=width, dropout=dropout).to(device)

def loss_fn(D, out, y):  # out: (B, k, d_out). Each submodel is trained on the same target.
    K = out.shape[1]
    if D['task'] == 'regression':
        return F.mse_loss(out[..., 0], y[:, None].expand(-1, K))
    return F.cross_entropy(out.flatten(0, 1), y.repeat_interleave(K))

@torch.no_grad()
def predict(model, D, part, bs=4096):
    model.eval(); P = D[part]; n = len(P['y']); preds = []
    for i in range(0, n, bs):
        xn = P['x_num'][i:i+bs] if P['x_num'] is not None else None
        xc = P['x_cat'][i:i+bs] if P['x_cat'] is not None else None
        o = model(xn, xc).float()
        preds.append(o[..., 0].mean(1) if D['task'] == 'regression' else torch.softmax(o, -1).mean(1))
    return torch.cat(preds)

def metric(D, part, pred):
    """Returns (raw metric, higher-is-better score). Regression: RMSE. Classification: accuracy."""
    y = D[part]['y']
    if D['task'] == 'regression':
        rmse = ((pred - y) ** 2).mean().sqrt().item() * D['y_std']
        return rmse, -rmse
    acc = (pred.argmax(1) == y).float().mean().item()
    return acc, acc

def sync():
    if device.type == 'cuda': torch.cuda.synchronize()

def train_one(D, k, depth, width, lr, seed, dropout=DROPOUT, wd=WEIGHT_DECAY, trial=None):
    torch.manual_seed(seed); np.random.seed(seed)
    model = build_model(D, k, depth, width, dropout)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    tr = D['train']; n = len(tr['y']); bs = batch_size_for(n)
    if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()

    best_val, best_state, bad, epoch = -math.inf, None, 0, 0
    sync(); t0 = time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for idx in torch.randperm(n, device=device).split(bs):
            xn = tr['x_num'][idx] if tr['x_num'] is not None else None
            xc = tr['x_cat'][idx] if tr['x_cat'] is not None else None
            loss = loss_fn(D, model(xn, xc), tr['y'][idx])
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        _, val_score = metric(D, 'val', predict(model, D, 'val'))
        if trial is not None:                       # pruning hook
            trial.report(val_score, epoch)
            if trial.should_prune(): raise optuna.TrialPruned()
        if val_score > best_val:
            best_val, bad = val_score, 0
            best_state = {kk: v.detach().clone() for kk, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: break
    sync(); train_time = time.time() - t0

    model.load_state_dict(best_state)
    sync(); t1 = time.time(); test_pred = predict(model, D, 'test'); sync()
    infer_obj_per_s = len(D['test']['y']) / max(time.time() - t1, 1e-9)
    test_raw, _ = metric(D, 'test', test_pred)
    return {'val_score': best_val, 'test_raw': test_raw, 'train_time_s': train_time, 'epochs': epoch,
            'n_params': sum(p.numel() for p in model.parameters()),
            'infer_obj_per_s': infer_obj_per_s,
            'peak_mem_mb': torch.cuda.max_memory_allocated() / 2**20 if device.type == 'cuda' else np.nan}

In [ ]:
def make_objective(D, name, task):
    def objective(trial):
        k       = 2 ** trial.suggest_int('log2_k', 2, 6)
        depth   = trial.suggest_int('depth', 1, 6)
        width   = 2 ** trial.suggest_int('log2_width', 6, 10)
        lr      = trial.suggest_float('lr', 3e-4, 1e-2, log=True)
        dropout = trial.suggest_float('dropout', 0.0, 0.4)
        wd      = trial.suggest_float('wd', 1e-6, 1e-2, log=True)
        t_cfg = time.time()
        r = train_one(D, k, depth, width, lr, seed=0, dropout=dropout, wd=wd, trial=trial)
        row = {'dataset': name, 'task': task, 'k': k, 'depth': depth, 'width': width,
               'best_lr': lr, 'dropout': dropout, 'weight_decay': wd,
               'higher_better': task != 'regression',
               'test_mean': r['test_raw'], 'test_std': 0.0, 'val_mean': r['val_score'],
               'train_time_s': r['train_time_s'], 'epochs': r['epochs'], 'n_params': r['n_params'],
               'infer_obj_per_s': r['infer_obj_per_s'], 'peak_mem_mb': r['peak_mem_mb'],
               'tune_time_s': time.time() - t_cfg}
        pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', header=not os.path.exists(RESULTS_CSV), index=False)
        print(f'  k={k:<3} depth={depth} width={width:<4} lr={lr:.1e} do={dropout:.2f} wd={wd:.0e}  '
              f'val={r["val_score"]:.4f} test={r["test_raw"]:.4f}  {r["train_time_s"]:.1f}s')
        return r['val_score']          # optimise on validation only, never test
    return objective

meta_rows = pd.read_csv(META_CSV).to_dict('records') if os.path.exists(META_CSV) else []

for name, source, task in DATASETS:
    study = optuna.create_study(
        study_name=f'tabm_{name}', storage=STORAGE, load_if_exists=True, direction='maximize',
        sampler=optuna.samplers.TPESampler(n_startup_trials=N_STARTUP, multivariate=True, seed=0),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=N_STARTUP, n_warmup_steps=10))
    n_left = N_TRIALS - sum(t.state.is_finished() for t in study.trials)
    if n_left <= 0:
        print(f'[{name}] already done'); continue
    try:
        D, meta = prepare(name, source, task)
    except Exception as e:
        print(f'[{name}] SKIPPED, could not load: {e}'); continue
    if name not in {m['dataset'] for m in meta_rows}:
        meta_rows.append(meta); pd.DataFrame(meta_rows).to_csv(META_CSV, index=False)
    print(f'\n[{name}] task={task} train={meta["n_train"]} num={meta["n_num"]} cat={meta["n_cat"]}  ({n_left} trials left)')
    if not study.trials:
        study.enqueue_trial(DEFAULT_PARAMS)    # trial 0 = default TabM, so Q2 still has a reference
    study.optimize(make_objective(D, name, task), n_trials=n_left)
    del D; torch.cuda.empty_cache() if device.type == 'cuda' else None
print('\nDone.')


[california] task=regression train=10000 num=8 cat=0  (30 trials left)
  k=32  depth=3 width=512  lr=2.0e-03 do=0.10 wd=3e-04  val=-0.4991 test=0.4859  22.7s
  k=16  depth=5 width=512  lr=2.0e-03 do=0.17 wd=4e-04  val=-0.4941 test=0.4847  19.5s
  k=16  depth=6 width=1024 lr=1.2e-03 do=0.32 wd=1e-04  val=-0.4937 test=0.4874  92.2s
  k=16  depth=6 width=64   lr=4.1e-04 do=0.01 wd=2e-03  val=-0.5622 test=0.5490  12.0s
  k=32  depth=6 width=1024 lr=4.9e-03 do=0.18 wd=1e-03  val=-0.4999 test=0.4888  175.2s
  k=4   depth=4 width=64   lr=8.2e-03 do=0.21 wd=5e-05  val=-0.5406 test=0.5290  8.9s
  k=8   depth=5 width=256  lr=2.2e-03 do=0.01 wd=3e-04  val=-0.4909 test=0.4739  10.5s
  k=32  depth=4 width=1024 lr=3.3e-03 do=0.14 wd=6e-05  val=-0.4806 test=0.4691  110.3s
  k=64  depth=3 width=128  lr=8.0e-03 do=0.04 wd=1e-05  val=-0.4897 test=0.4760  10.3s
  k=64  depth=4 width=1024 lr=5.5e-03 do=0.01 wd=1e-06  val=-0.4749 test=0.4613  217.7s
  k=32  depth=4 width=512  lr=3.0e-03 do=0.01 wd=9e-06  

In [ ]:
df = pd.read_csv(RESULTS_CSV)
meta = pd.read_csv(META_CSV).set_index('dataset')
CFG = ['k', 'depth', 'width']

# Higher-is-better score; regret = % worse than the best config on that dataset
df['score'] = np.where(df.higher_better, df.test_mean, -df.test_mean)
best = df.groupby('dataset').score.transform('max')
df['regret_pct'] = (best - df.score) / best.abs() * 100
# "Tied with best" = within the best config's seed std (same spirit as the paper's rank rule)
best_std = df.loc[df.groupby('dataset').score.idxmax()].set_index('dataset').test_std
df['tied_best'] = df.score >= best - df.dataset.map(best_std)

# ---- Q1: does the best config differ across datasets? ----
print('=== Q1: Best config per dataset ===')
b = df.loc[df.groupby('dataset').score.idxmax(), ['dataset'] + CFG + ['best_lr', 'dropout', 'weight_decay', 'test_mean', 'test_std']]
b['n_tied_configs'] = b.dataset.map(df.groupby('dataset').tied_best.sum())
b = b.join(meta[['n_train', 'n_num', 'n_cat']], on='dataset').sort_values('n_train')
print(b.to_string(index=False))

# ---- Q2: how much does the default config lose? ----
print('\n=== Q2: Regret of default TabM (k=32, depth=3, width=512, lr=2e-3, dropout=0.1, wd=3e-4) ===')
dd = df[(df.k == 32) & (df.depth == 3) & (df.width == 512) &
        np.isclose(df.best_lr, 2e-3) & np.isclose(df.dropout, 0.1) & np.isclose(df.weight_decay, 3e-4)]
d = dd[['dataset', 'regret_pct', 'tied_best']]
print(d.to_string(index=False))
print(f'Mean regret of default: {d.regret_pct.mean():.2f}%')

# ---- Cost side: how expensive are the configs? ----
print('\n=== Mean train time (s) and params per config ===')
print(df.groupby(CFG)[['train_time_s', 'n_params', 'peak_mem_mb']].mean().round(1).to_string())

=== Q1: Best config per dataset ===
       dataset  k  depth  width  best_lr  dropout  weight_decay    test_mean  test_std  n_tied_configs  n_train  n_num  n_cat
       phoneme  8      5    256 0.003452 0.013664      0.000136     0.895467       0.0               1     3458      5      0
      satimage 64      3   1024 0.003269 0.145993      0.000061     0.930016       0.0               1     4115     36      0
MagicTelescope 32      3    512 0.002000 0.100000      0.000300     0.896162       0.0               1    10000     10      0
         adult 32      1   1024 0.008474 0.226972      0.000003     0.851200       0.0               2    10000      6      8
      diamonds 16      5    512 0.002027 0.169462      0.000383   591.651466       0.0               1    10000      6      3
    california 64      3    512 0.006838 0.015739      0.000001     0.460835       0.0               1    10000      8      0
     house_16H 16      5    512 0.005141 0.079858      0.000560 30445.140656      